# Cross-Sectional Momentum on S&P 500

**Research question:** Do past winners continue to outperform past losers in the S&P 500?

**Hypothesis:** Stocks with the highest returns over the past 12 months will outperform stocks with the lowest returns over the following month.

In [49]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

## 1. Data Collection
Download daily adjusted close prices for all S&P 500 constituents.

In [50]:
response = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
data_frame = pd.read_csv(pd.io.common.StringIO(response.text))

list_sp500 = data_frame["Symbol"].tolist()
print(list_sp500[:10])

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## 2. Download Historical Prices
We download daily adjusted close prices for all S&P 500 constituents since 2010. Adjusted close accounts for dividends and stock splits, ensuring consistent return calculations.

In [51]:
# Fix Yahoo Finance ticker format
list_sp500 = [t.replace('.', '-') for t in list_sp500]

data_candles = yf.download(list_sp500, "2010-01-01", auto_adjust=True)

print(data_candles.shape)
print(data_candles.head())

[*********************100%***********************]  503 of 503 completed


(4236, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
Date                                                                        
2010-01-04  19.743517  6.400960  NaN  NaN  17.968187  7.601905  30.610764   
2010-01-05  19.529055  6.412028  NaN  NaN  17.823019  7.576549  30.799948   
2010-01-06  19.459669  6.310034  NaN  NaN  17.921999  7.543795  31.127382   
2010-01-07  19.434437  6.298370  NaN  NaN  18.070475  7.499420  31.098278   
2010-01-08  19.428125  6.340245  NaN  NaN  18.162846  7.484628  30.974573   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
Date                                         ...                              
2010-01-04  37.090000  21.703714  20.029289  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669449  20.137486  ...  1724

In [52]:
price = data_candles["Close"]
print(price.head())

Ticker              A      AAPL  ABBV  ABNB        ABT      ACGL        ACN  \
Date                                                                          
2010-01-04  19.743517  6.400960   NaN   NaN  17.968187  7.601905  30.610764   
2010-01-05  19.529055  6.412028   NaN   NaN  17.823019  7.576549  30.799948   
2010-01-06  19.459669  6.310034   NaN   NaN  17.921999  7.543795  31.127382   
2010-01-07  19.434437  6.298370   NaN   NaN  18.070475  7.499420  31.098278   
2010-01-08  19.428125  6.340245   NaN   NaN  18.162846  7.484628  30.974573   

Ticker           ADBE        ADI        ADM  ...        WY       WYNN  \
Date                                         ...                        
2010-01-04  37.090000  21.703714  20.029289  ...  9.157349  40.493740   
2010-01-05  37.700001  21.669449  20.137486  ...  9.351743  42.956528   
2010-01-06  37.619999  21.628338  20.086565  ...  9.248339  42.393074   
2010-01-07  36.889999  21.457003  19.876534  ...  9.206982  43.298412   
2010-01-

## 3. Compute Monthly Returns
We resample daily prices to monthly frequency and compute log returns. Log returns are additive over time, which simplifies the calculation of cumulative multi-month returns needed for the momentum signal.

In [53]:
monthly_price = price.resample('ME').last()
monthly_log = np.log(monthly_price)
monthly_yield = monthly_log.diff()
print(monthly_yield.head())

Ticker             A      AAPL  ABBV  ABNB       ABT      ACGL       ACN  \
Date                                                                       
2010-01-31       NaN       NaN   NaN   NaN       NaN       NaN       NaN   
2010-02-28  0.115441  0.063347   NaN   NaN  0.024997  0.033538 -0.025199   
2010-03-31  0.089049  0.138431   NaN   NaN -0.029920  0.030223  0.048349   
2010-04-30  0.052949  0.105280   NaN   NaN -0.020889 -0.008826  0.048235   
2010-05-31 -0.113792 -0.016256   NaN   NaN -0.072966 -0.027634 -0.151100   

Ticker          ADBE       ADI       ADM  ...        WY      WYNN       XEL  \
Date                                      ...                                 
2010-01-31       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2010-02-28  0.070231  0.081184 -0.015513  ...  0.012454  0.026945  0.001443   
2010-03-31  0.020566 -0.007693 -0.015791  ...  0.113814  0.176352  0.030028   
2010-04-30 -0.051338  0.037792 -0.033782  ...  0.090730  0.151566  0.025

## 4. Momentum Signal
For each stock, we compute the cumulative log return over the past 12 months. This is our momentum score: stocks with high past returns are "winners", stocks with low past returns are "losers".

In [54]:
annual_yield = monthly_yield.rolling(12).sum()
print(annual_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-07-31  0.194645  0.401541  0.313912  0.134814 -0.154333  0.155411   
2026-08-31  0.208504  0.314775  0.228358  0.339085 -0.161191  0.069015   
2026-09-30  0.300521  0.272078  0.152577  0.279972 -0.281140  0.017806   
2026-10-31  0.185450  0.246367  0.331414  0.261683 -0.127337  0.208884   
2026-11-30  0.150885  0.228576  0.287859  0.410496 -0.085129  0.164200   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-07-31 -0.455720 -0.356567  0.505358  0.411562  ...  0.033970 -0.084215   
2026-08-31 -0.294409 -0.197439  0.378678  0.289418  ... -0.074988 -0.318965   
2026-09-30 -0.275865 -0.385371  0.491524  0.313044  ... -0.227574 -0.469941   
2026-10-31  0.060416 -0.307476  0.589903  0.268296  ... -0.086614 -0.456775   
2026-11

## 5. Portfolio Construction
Each month, we rank all stocks by their 12-month momentum score and sort them into deciles (10 groups). Decile 1 contains the biggest losers, decile 10 the biggest winners.

In [55]:
ranked_yield = annual_yield.rank(axis=1, pct=True)
print(ranked_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-07-31  0.593186  0.827655  0.731463  0.507014  0.170341  0.545090   
2026-08-31  0.651303  0.775551  0.683367  0.809619  0.154309  0.450902   
2026-09-30  0.801603  0.767535  0.649299  0.783567  0.136273  0.456914   
2026-10-31  0.594000  0.690000  0.804000  0.712000  0.152000  0.630000   
2026-11-30  0.528000  0.662000  0.766000  0.918000  0.162000  0.550000   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-07-31  0.042084  0.072144  0.885772  0.837675  ...  0.358717  0.216433   
2026-08-31  0.076152  0.132265  0.841683  0.759519  ...  0.226453  0.056112   
2026-09-30  0.140281  0.080160  0.909820  0.817635  ...  0.174349  0.058116   
2026-10-31  0.386000  0.076000  0.954000  0.718000  ...  0.190000  0.048000   
2026-11

In [56]:
deciles = np.ceil(ranked_yield * 10)
print(deciles.tail())

Ticker        A  AAPL  ABBV  ABNB  ABT  ACGL  ACN  ADBE   ADI  ADM  ...   WY  \
Date                                                                ...        
2026-07-31  6.0   9.0   8.0   6.0  2.0   6.0  1.0   1.0   9.0  9.0  ...  4.0   
2026-08-31  7.0   8.0   7.0   9.0  2.0   5.0  1.0   2.0   9.0  8.0  ...  3.0   
2026-09-30  9.0   8.0   7.0   8.0  2.0   5.0  2.0   1.0  10.0  9.0  ...  2.0   
2026-10-31  6.0   7.0   9.0   8.0  2.0   7.0  4.0   1.0  10.0  8.0  ...  2.0   
2026-11-30  6.0   7.0   8.0  10.0  2.0   6.0  5.0   1.0  10.0  6.0  ...  3.0   

Ticker      WYNN  XEL   XOM  XYL  XYZ  YUM  ZBH  ZBRA  ZTS  
Date                                                        
2026-07-31   3.0  5.0   8.0  2.0  4.0  5.0  4.0   2.0  1.0  
2026-08-31   1.0  5.0   9.0  2.0  4.0  5.0  3.0   6.0  1.0  
2026-09-30   1.0  3.0   9.0  1.0  5.0  4.0  3.0   8.0  1.0  
2026-10-31   1.0  4.0  10.0  1.0  4.0  6.0  3.0   8.0  1.0  
2026-11-30   1.0  3.0  10.0  1.0  8.0  5.0  5.0   8.0  1.0  

[5 rows x 5

## 6. Backtest
Each month, we go long the top decile (winners) and short the bottom decile (losers). We compute the equal-weighted return of each portfolio and track the long-short spread over time.

In [57]:
signal = deciles.shift(1)
long_ret = monthly_yield.where(signal == 10).mean(axis=1)
short_ret = monthly_yield.where(signal == 1).mean(axis=1)
strategy_ret = long_ret - short_ret
print(strategy_ret.tail())

Date
2026-07-31   -0.262365
2026-08-31   -0.068201
2026-09-30    0.176429
2026-10-31   -0.157845
2026-11-30   -0.106373
Freq: ME, dtype: float64


## 7. Performance Analysis
We compute key performance metrics for the long-short momentum strategy: annualized return, Sharpe ratio, and win rate.

In [58]:
monthly_ret = strategy_ret.mean()
standard_deviation = strategy_ret.std()
positive_months = (strategy_ret > 0).sum()
negative_months = (strategy_ret < 0).sum()
print(monthly_ret)
print(standard_deviation)
print(positive_months)
print(negative_months)

0.005022015849239592
0.06219871647385579
102
88


In [59]:
annual_ret = monthly_ret * 12
annual_std = standard_deviation * np.sqrt(12)
sharpe_ratio = annual_ret / annual_std
print(f"Annual ret: {annual_ret:.3f}")
print(f"Annual std: {annual_std:.3f}")
print(f"Sharpe ratio: {sharpe_ratio:.3f}")

Annual ret: 0.060
Annual std: 0.215
Sharpe ratio: 0.280


## 8. Robustness Checks
### 8.1 Alpha Decay Test
We split the sample into two sub-periods to test whether the momentum premium has weakened over time.

In [60]:
early_ret = strategy_ret.loc[strategy_ret.index.year <= 2018]
late_ret = strategy_ret.loc[strategy_ret.index.year > 2018]

monthly_early_ret = early_ret.mean()
monthly_late_ret = late_ret.mean()

annual_early_ret = monthly_early_ret * 12
annual_late_ret = monthly_late_ret * 12

standard_deviation_early = early_ret.std()
standard_deviation_late = late_ret.std()

annual_std_early = standard_deviation_early * np.sqrt(12)
annual_std_late = standard_deviation_late * np.sqrt(12)

early_sharpe_ratio = annual_early_ret / annual_std_early
late_sharpe_ratio = annual_late_ret / annual_std_late

print(f"Early period (2011-2018) Sharpe: {early_sharpe_ratio:.3f}")
print(f"Late period (2019-2026) Sharpe: {late_sharpe_ratio:.3f}")

Early period (2011-2018) Sharpe: 0.196
Late period (2019-2026) Sharpe: 0.343


### 8.2 Lookback Window: 12-0 vs 12-1
Academic literature (Jegadeesh & Titman, 1993) recommends skipping the most recent month when computing the momentum signal (12-1 instead of 12-0) to avoid short-term reversal contamination.

We tested this by replacing the 12-month rolling sum with a shifted 11-month sum. Results:
- **12-0 Sharpe: 0.280**
- **12-1 Sharpe: 0.245**

The 12-1 variant underperforms in our sample, suggesting the short-term reversal effect is not strong enough to degrade the 12-0 signal. We retain the 12-0 specification for the rest of the analysis.

### 8.3 Sector Analysis
We investigate whether the momentum premium is driven by a specific sector or is broadly distributed. If momentum alpha comes from only one sector, it may reflect sector rotation rather than genuine stock-level momentum.

In [73]:
sector_map = data_frame.set_index('Symbol')['GICS Sector'].to_dict()

last_month = deciles.iloc[-1]

winners = last_month[last_month == 10]
winner_sectors = winners.index.map(sector_map)
winner_sectors_count = winner_sectors.value_counts()

loosers = last_month[last_month == 1]
looser_sectors = loosers.index.map(sector_map)
looser_sectors_count = looser_sectors.value_counts()

print(f"Winners : {winner_sectors_count}")
print(f"Loosers : {looser_sectors_count}")

Winners : Ticker
Information Technology    17
Health Care               10
Energy                     8
Consumer Discretionary     5
Industrials                5
Financials                 2
Real Estate                1
Materials                  1
Consumer Staples           1
Name: count, dtype: int64
Loosers : Ticker
Information Technology    9
Consumer Discretionary    8
Industrials               8
Communication Services    6
Health Care               6
Financials                4
Utilities                 3
Materials                 2
Consumer Staples          2
Real Estate               1
Energy                    1
Name: count, dtype: int64


In [90]:
list_of_sector_winner = []
list_of_sector_looser = []

for date, line in deciles.iterrows():
    winners_backtest = line[line == 10]
    winner_sectors_backtest = winners_backtest.index.map(sector_map)
    winner_sectors_count_backtest = winner_sectors_backtest.value_counts()
    list_of_sector_winner.append(winner_sectors_count_backtest)

sector_df_winner = pd.DataFrame(list_of_sector_winner).fillna(0)
print(sector_df_winner.mean().sort_values(ascending=False))

for date, line in deciles.iterrows():
    loosers_backtest = line[line == 1]
    looser_sectors_backtest = loosers_backtest.index.map(sector_map)
    looser_sectors_count_backtest = looser_sectors_backtest.value_counts()
    list_of_sector_looser.append(looser_sectors_count_backtest)

sector_df_looser = pd.DataFrame(list_of_sector_looser).fillna(0)
print(sector_df_looser.mean().sort_values(ascending=False))

print(data_frame['GICS Sector'].value_counts())

Ticker
Information Technology    10.980296
Industrials                6.162562
Consumer Discretionary     6.157635
Health Care                5.315271
Financials                 4.226601
Energy                     2.965517
Communication Services     2.399015
Consumer Staples           1.847291
Materials                  1.768473
Real Estate                1.600985
Utilities                  1.162562
dtype: float64
Ticker
Information Technology    7.147783
Health Care               5.384236
Consumer Discretionary    5.325123
Industrials               4.665025
Financials                4.581281
Energy                    3.847291
Consumer Staples          2.768473
Real Estate               2.743842
Materials                 2.522167
Communication Services    2.310345
Utilities                 2.246305
dtype: float64
GICS Sector
Industrials               83
Information Technology    76
Financials                76
Health Care               60
Consumer Discretionary    47
Consumer Staples  

***Findings:*** The momentum signal shows a sector bias. Information Technology is over-represented in the winner portfolio

### 8.4 Lookback Period Sensitivity
We test whether the momentum signal is sensitive to the lookback window by comparing 3-month, 6-month and 12-month cumulative returns as the signal.